In [0]:
%pip install torchinfo

In [0]:
import torch
import mlflow
from mlflow.models.signature import infer_signature
from torch import nn
from torchinfo import summary

torch.__version__


# Data Preperation & Loading

In [0]:
# Create some data for a linear regression problem
weight = 0.7
bias = 0.3

start = 0
end = 1
step = 0.02

data_x = torch.arange(start, end, step).unsqueeze(dim=1)
label_y = weight * data_x + bias

# create a train / test split

train_split = int(0.8 * len(data_x))
x_train, y_train = data_x[:train_split], label_y[:train_split]
x_test, y_test = data_x[train_split:], label_y[train_split:]

len(x_train), len(y_train), len(x_test), len(y_test)


In [0]:
# visualise the data
import matplotlib.pyplot as plt

def plot_predictions(train_data,
                     train_labels,
                     test_data,
                     test_labels,
                     predictions=None):
    """
    Plots training data, test data and compares predictions.
    """
    plt.figure(figsize=(10, 7))
    # Plot training data in blue
    plt.scatter(train_data, train_labels, c="b", s=4, label="Training data")
    # Plot test data in green
    plt.scatter(test_data, test_labels, c="g", s=4, label="Testing data")
    if predictions is not None:
        # Plot the predictions in red (predictions were made on the test data)
        plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")
    # Show the legend
    plt.legend(prop={"size": 14})
    plt.show()

plot_predictions(
    train_data=x_train,
    train_labels=y_train,
    test_data=x_test,
    test_labels=y_test
)

##  Build the Model

Standard linear regression model using PyTorch


In [0]:
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(in_features=1, out_features=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
      return self.linear(x)

def get_device():
    # move the model to the GPU if available
    device = "cpu"
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.mps.is_available():
        device = "mps"
    device = torch.device(device)
    return device


In [0]:
hyper_params = {
  "manual_seed": 42,
  "epochs": 200,
  "learning_rate": 0.01,
  # "batch_size": 64,
  "optimizer": "SGD",
  "model_type": "MLP"
}

with mlflow.start_run():
  # log the parameters
  mlflow.log_params(hyper_params)

  # create and prepare the model
  torch.manual_seed(hyper_params["manual_seed"])
  model = LinearRegressionModel()
  model.to(get_device())
  loss_fn = nn.L1Loss() # Mean Absolute Error (MAE)
  optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01) # stocastic gradient descent

  # Log model architecture
  with open("model_summary.txt", "w") as f:
    f.write(str(summary(model, input_size=[1])))
  mlflow.log_artifact("model_summary.txt")


  for epoch in range(hyper_params["epochs"]):
    # set the model to training mode
    model.train()

    # Forward Pass
    y_pred = model(x_train)
    loss = loss_fn(y_pred, y_train)
    optimizer.zero_grad()

    # Backward Pass
    loss.backward()
    optimizer.step()

    # set the model to evaluation mode for testing
    model.eval()

    with torch.inference_mode():
      # Do the forward pass
      test_pred = model(x_test)
      # Calculate the loss
      test_loss = loss_fn(test_pred, y_test.type(torch.float)) # predictions come in torch.float datatype
  
    mlflow.log_metrics(
      {
        "train_loss": loss, 
        "test_loss": test_loss
      }, step=epoch
    )

    # Print out what's happening
    if epoch % 10 == 0:
      print(f"Epoch: {epoch} | MAE Train Loss: {loss} | MAE Test Loss: {test_loss} ")

  model_info = mlflow.pytorch.log_model(model, name="pytorch_intro_lr", input_example=float(0.1))

  print(str(model_info))



In [0]:
# Note that the UC model name follows the pattern
# <catalog_name>.<schema_name>.<model_name>, corresponding to
# the catalog, schema, and registered model name
# in Unity Catalog under which to create the version
# The registered model will be created if it doesn't already exist,
# and the model version will contain all parameters and metrics
# logged with the corresponding MLflow Logged Model.
mlflow.register_model(model_info.model_uri, "dev_hub.machine_learning.pytorch_intro_lr")

In [0]:
# # Load and use the model from a URI.
loaded_model:LinearRegressionModel = mlflow.pytorch.load_model("models:/dev_hub.machine_learning.pytorch_intro_lr/2")


In [0]:
#make predictions
# 1. Put the loaded model into evaluation mode
loaded_model.eval()

# 2. Use the inference mode context manager to make predictions
with torch.inference_mode():
    loaded_model_preds = loaded_model(x_test) # perform a forward pass on the test data with the loaded model


loaded_model_preds

In [0]:
# tag model

from mlflow import MlflowClient
client = MlflowClient()

# create "Champion" alias for version 1 of model "prod.ml_team.iris_model"
client.set_registered_model_alias("dev_hub.machine_learning.pytorch_intro_lr", "champion", 2)